# Prepare auxiliary flux data

This notebook prepares environmental and chamber-geometry inputs for a later flux calculation. It does not calculate dC/dt or flux.

Workflow:

1. Load CSV files or a project JSON folder.
2. Map the timestamp, measurement, environmental, chamber, and geometry columns.
3. Set valid ranges, prepare the data, and review per-measurement quality and before/after plots.
4. Export one auxiliary NetCDF file per chamber and measurement-start day.

A measurement that has no valid pressure, temperature, or relative-humidity anchor is still exported. Its missing values remain unresolved and its status is invalid_auxiliary.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import sys
import traceback
import urllib.parse

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import clear_output, display

# Make the package import work whether Jupyter starts in the repository root
# or in this notebook folder.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "soilgasflux_fcs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "soilgasflux_fcs").exists():
    raise RuntimeError("Could not locate the repository root containing soilgasflux_fcs.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from soilgasflux_fcs import json_reader
from soilgasflux_fcs.models import mole_fraction_water_vapor

DEFAULT_AREA_CM2 = 314.0
DEFAULT_VOLUME_CM3 = 6283.0
DEFAULT_OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "processing" / "output" / "auxiliary"
SCHEMA_VERSION = "1.0"

ENVIRONMENT_SPECS = {
    "pressure_kpa": {
        "label": "Pressure",
        "unit": "kPa",
        "source": "pressure_source",
        "flag": "pressure_quality_flag",
    },
    "temperature_c": {
        "label": "Temperature",
        "unit": "degree_Celsius",
        "source": "temperature_source",
        "flag": "temperature_quality_flag",
    },
    "relative_humidity_percent": {
        "label": "Relative humidity",
        "unit": "percent",
        "source": "relative_humidity_source",
        "flag": "relative_humidity_quality_flag",
    },
}

FLAG_MEANINGS = {
    "environment": "0=original_valid, 1=interpolated_or_edge_filled, 2=invalid_unresolved",
    "water_vapor": "0=derived_from_original, 1=derived_from_filled, 2=invalid",
    "geometry": "0=mapped_valid, 1=default_supplied, 2=invalid",
    "measurement": "0=valid_original, 1=usable_with_fills_or_defaults, 2=invalid_auxiliary",
}


## Load and normalize input

CSV mode accepts one file or a folder of files. JSON mode accepts a folder containing the project raw-data JSON files used by the other processing notebooks.


In [2]:
def sanitize_input_path(input_path):
    text = str(input_path or "").strip()
    if not text:
        raise FileNotFoundError("Input path is empty. Paste a file or folder path before loading.")

    if text.startswith(("Path(", "pathlib.Path(")) and text.endswith(")"):
        text = text[text.find("(") + 1:-1].strip()

    for _ in range(2):
        if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
            text = text[1:-1].strip()

    parsed = urllib.parse.urlparse(text)
    if parsed.scheme == "file":
        text = urllib.parse.unquote(parsed.path)
    if text.startswith("Users/"):
        text = "/" + text
    return Path(text).expanduser()


def find_matching_folders(input_path, folder_name=None):
    root = sanitize_input_path(input_path)
    if not root.exists():
        raise FileNotFoundError(f"Input path does not exist: {root}")
    if not root.is_dir():
        if folder_name is None:
            return []
        raise NotADirectoryError(
            f"Folder filtering requires a root folder, not a file: {root}"
        )
    if folder_name is None:
        return [root]

    name = str(folder_name).strip()
    if not name:
        raise ValueError("Enter a folder name when 'Filter folder' is enabled.")
    if (
        name in {".", ".."}
        or any(character in name for character in ("/", "\\", "*", "?", "[", "]"))
    ):
        raise ValueError(
            "Folder filter must be one exact folder name without path separators or wildcards."
        )

    candidates = [root] if root.name == name else []
    candidates.extend(
        candidate
        for candidate in root.rglob("*")
        if candidate.is_dir() and candidate.name == name
    )
    if not candidates:
        raise FileNotFoundError(
            f"No folder named {name!r} was found under {root}."
        )

    matches = []
    for candidate in sorted(set(candidates), key=lambda item: (len(item.parts), str(item))):
        if any(parent == candidate or parent in candidate.parents for parent in matches):
            continue
        matches.append(candidate)
    return matches


def find_csv_files(input_path, pattern="*.csv", folder_name=None):
    path = sanitize_input_path(input_path)
    if path.is_file():
        if folder_name is not None:
            raise NotADirectoryError(
                f"Folder filtering requires a root folder, not a file: {path}"
            )
        if path.suffix.lower() != ".csv":
            raise ValueError(f"CSV mode requires a CSV file, received: {path.name}")
        return [path]
    if path.is_dir():
        search_folders = find_matching_folders(path, folder_name)
        if folder_name is None:
            files = sorted(file for file in path.glob(pattern) if file.is_file())
        else:
            files = sorted({
                file
                for folder in search_folders
                for file in folder.rglob(pattern)
                if file.is_file()
            })
        if not files:
            scope = path if folder_name is None else f"folders named {str(folder_name).strip()!r} under {path}"
            raise FileNotFoundError(f"No CSV files matched {pattern!r} in {scope}.")
        return files
    raise FileNotFoundError(f"Input path does not exist: {path}")


def find_json_files(folder_path, min_size_bytes=5000, folder_name=None):
    folder = sanitize_input_path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"JSON folder does not exist: {folder}")
    if not folder.is_dir():
        raise NotADirectoryError(f"JSON input must be a folder: {folder}")

    search_folders = find_matching_folders(folder, folder_name)
    all_json = sorted({
        file
        for search_folder in search_folders
        for file in search_folder.rglob("*.json")
        if file.is_file()
    })
    usable = [file for file in all_json if file.stat().st_size >= min_size_bytes]
    ignored = [file for file in all_json if file.stat().st_size < min_size_bytes]
    scope = folder if folder_name is None else f"folders named {str(folder_name).strip()!r} under {folder}"
    if not all_json:
        raise FileNotFoundError(f"No JSON files were found in {scope}.")
    if not usable:
        raise FileNotFoundError(
            f"All {len(all_json)} JSON file(s) in {scope} were smaller than {min_size_bytes} bytes."
        )
    return usable, ignored


def emit_progress(progress_callback, completed, total, message):
    if progress_callback is not None:
        progress_callback(int(completed), max(int(total), 1), str(message))


def load_csv_input(
    input_path, pattern="*.csv", delimiter=",", folder_name=None,
    progress_callback=None,
):
    path = sanitize_input_path(input_path)
    files = find_csv_files(path, pattern, folder_name=folder_name)
    frames = []
    total = len(files)
    for index, file in enumerate(files, start=1):
        emit_progress(progress_callback, index - 1, total, f"Reading {file.name}")
        frame = pd.read_csv(file, sep=delimiter)
        frame["__source_file"] = str(file)
        frames.append(frame)
        emit_progress(progress_callback, index, total, f"Read {index}/{total} CSV files")
    raw = pd.concat(frames, ignore_index=True)
    raw.attrs["source_files"] = [str(file) for file in files]
    raw.attrs["ignored_files"] = []
    raw.attrs["folder_filter"] = None if folder_name is None else str(folder_name).strip()
    raw.attrs["matched_folders"] = (
        []
        if folder_name is None
        else [str(folder) for folder in find_matching_folders(path, folder_name)]
    )
    return raw


def load_json_input(
    folder_path, min_size_bytes=5000, folder_name=None, progress_callback=None
):
    folder = sanitize_input_path(folder_path)
    search_folders = find_matching_folders(folder, folder_name)
    files, ignored = find_json_files(folder, min_size_bytes, folder_name=folder_name)
    load_folders = [
        search_folder
        for search_folder in search_folders
        if any(search_folder == file.parent or search_folder in file.parents for file in files)
    ]
    frames = []
    try:
        total = len(load_folders)
        for index, search_folder in enumerate(load_folders, start=1):
            emit_progress(
                progress_callback, index - 1, total,
                f"Reading JSON folder {search_folder.name}",
            )
            frames.append(json_reader.Initializer(search_folder).prepare_rawdata())
            emit_progress(
                progress_callback, index, total,
                f"Read {index}/{total} JSON folders",
            )
    except (KeyError, ValueError) as exc:
        raise ValueError(
            "The selected JSON folders could not be read as project raw data. "
            "Each usable JSON file needs a raw_data section."
        ) from exc
    if not frames:
        raise ValueError("The selected JSON folders produced no data rows.")
    raw = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0].reset_index(drop=True)
    raw.sort_values(by="datetime", inplace=True, ignore_index=True)
    if raw.empty:
        raise ValueError("The JSON folder produced no data rows.")
    raw["__source_file"] = raw.get("id", "").astype(str)
    raw.attrs["source_files"] = [str(file) for file in files]
    raw.attrs["ignored_files"] = [str(file) for file in ignored]
    raw.attrs["folder_filter"] = None if folder_name is None else str(folder_name).strip()
    raw.attrs["matched_folders"] = (
        [] if folder_name is None else [str(search_folder) for search_folder in search_folders]
    )
    return raw


def load_auxiliary_input(
    mode, input_path, pattern="*.csv", delimiter=",", folder_name=None,
    progress_callback=None,
):
    if mode == "CSV":
        return load_csv_input(
            input_path, pattern=pattern, delimiter=delimiter, folder_name=folder_name,
            progress_callback=progress_callback,
        )
    if mode == "JSON folder":
        return load_json_input(
            input_path, folder_name=folder_name, progress_callback=progress_callback
        )
    raise ValueError(f"Unsupported input mode: {mode}")


def guess_column(columns, candidates):
    normalized = {str(column).strip().lower(): column for column in columns}
    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    for column in columns:
        lowered = str(column).lower()
        if any(candidate.lower() in lowered for candidate in candidates):
            return column
    return ""


def _numeric_source(raw, column, label):
    if not column:
        raise ValueError(f"Select a source column for {label}.")
    return pd.to_numeric(raw[column], errors="coerce")


def normalize_auxiliary_input(
    raw,
    *,
    timestamp_col,
    measurement_id_col,
    pressure_col,
    pressure_unit,
    temperature_col,
    humidity_col,
    chamber_id_col="",
    constant_chamber_id="",
    area_col="",
    volume_col="",
    default_area_cm2=DEFAULT_AREA_CM2,
    default_volume_cm3=DEFAULT_VOLUME_CM3,
):
    if not timestamp_col:
        raise ValueError("An absolute timestamp column is required.")
    if not measurement_id_col:
        raise ValueError("A measurement ID column is required.")
    if not chamber_id_col and not str(constant_chamber_id).strip():
        raise ValueError("Map a chamber ID column or enter a constant chamber ID.")

    result = pd.DataFrame(index=raw.index)
    result["timestamp"] = pd.to_datetime(raw[timestamp_col], errors="coerce", utc=True)
    if result["timestamp"].isna().any():
        count = int(result["timestamp"].isna().sum())
        raise ValueError(f"The timestamp mapping produced {count} missing/invalid timestamp(s).")

    measurement = raw[measurement_id_col].astype("string").str.strip()
    missing_measurement = measurement.isna() | measurement.eq("")
    if missing_measurement.any():
        raise ValueError(f"Measurement ID is missing in {int(missing_measurement.sum())} row(s).")
    result["measurement_id"] = measurement.astype(str)

    if chamber_id_col:
        chamber = raw[chamber_id_col].astype("string").str.strip()
        missing_chamber = chamber.isna() | chamber.eq("")
        if missing_chamber.any():
            raise ValueError(f"Chamber ID is missing in {int(missing_chamber.sum())} row(s).")
        result["chamber_id"] = chamber.astype(str)
    else:
        result["chamber_id"] = str(constant_chamber_id).strip()

    pressure = _numeric_source(raw, pressure_col, "pressure")
    if pressure_unit == "Pa":
        pressure = pressure / 1000.0
    elif pressure_unit != "kPa":
        raise ValueError("Pressure unit must be Pa or kPa.")
    result["pressure_source"] = pressure.astype(float)
    result["temperature_source"] = _numeric_source(raw, temperature_col, "temperature").astype(float)
    result["relative_humidity_source"] = _numeric_source(raw, humidity_col, "relative humidity").astype(float)

    if area_col:
        result["chamber_area_cm2"] = pd.to_numeric(raw[area_col], errors="coerce").astype(float)
        result["area_default_used"] = False
    else:
        if not np.isfinite(default_area_cm2) or default_area_cm2 <= 0:
            raise ValueError("Default chamber area must be finite and greater than zero.")
        result["chamber_area_cm2"] = float(default_area_cm2)
        result["area_default_used"] = True

    if volume_col:
        result["chamber_volume_cm3"] = pd.to_numeric(raw[volume_col], errors="coerce").astype(float)
        result["volume_default_used"] = False
    else:
        if not np.isfinite(default_volume_cm3) or default_volume_cm3 <= 0:
            raise ValueError("Default chamber volume must be finite and greater than zero.")
        result["chamber_volume_cm3"] = float(default_volume_cm3)
        result["volume_default_used"] = True

    if "__source_file" in raw:
        result["source_file"] = raw["__source_file"].astype(str)
    else:
        result["source_file"] = ""

    result = result.sort_values(["chamber_id", "measurement_id", "timestamp"]).reset_index(drop=True)
    group_keys = ["chamber_id", "measurement_id"]
    result["measurement_start"] = result.groupby(group_keys)["timestamp"].transform("min")
    result["elapsed_seconds"] = (
        result["timestamp"] - result["measurement_start"]
    ).dt.total_seconds().astype(float)
    result["measurement_start_date"] = result["measurement_start"].dt.strftime("%Y-%m-%d")

    source_files = list(raw.attrs.get("source_files", []))
    ignored_files = list(raw.attrs.get("ignored_files", []))
    result.attrs["source_files"] = source_files
    result.attrs["ignored_files"] = ignored_files
    return result


def reject_reused_measurement_ids(normalized):
    spans = (
        normalized.groupby(["chamber_id", "measurement_id"])["timestamp"]
        .agg(["min", "max"])
        .reset_index()
    )
    spans["span_days"] = (spans["max"] - spans["min"]).dt.total_seconds() / 86400.0
    bad = spans[spans["span_days"] >= 1.0]
    if not bad.empty:
        examples = ", ".join(
            f"{row.chamber_id}/{row.measurement_id}"
            for row in bad.head(5).itertuples()
        )
        raise ValueError(
            "Measurement IDs must identify one closure within a chamber. "
            f"These IDs span 24 hours or more: {examples}"
        )


## Quality control and preview

Values outside the editable ranges are treated as invalid. Each environmental signal is filled independently within a chamber and measurement: linear interpolation for internal gaps and the nearest valid anchor at either edge. A signal with no valid anchor stays missing and receives flag 2.


In [3]:
def validate_ranges(ranges):
    for name, (lower, upper) in ranges.items():
        if not np.isfinite(lower) or not np.isfinite(upper):
            raise ValueError(f"{name} bounds must be finite.")
        if lower > upper:
            raise ValueError(f"{name} minimum must not exceed its maximum.")


def interpolate_environment_signal(elapsed, values, lower, upper):
    elapsed = np.asarray(elapsed, dtype=float)
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values) & (values >= lower) & (values <= upper)
    filled = np.full(values.shape, np.nan, dtype=float)
    flags = np.full(values.shape, 2, dtype=np.int8)

    if valid.any():
        order = np.argsort(elapsed, kind="stable")
        sorted_elapsed = elapsed[order]
        anchor_elapsed = elapsed[valid]
        anchor_values = values[valid]
        anchor_order = np.argsort(anchor_elapsed, kind="stable")
        interpolated = np.interp(
            sorted_elapsed,
            anchor_elapsed[anchor_order],
            anchor_values[anchor_order],
        )
        filled[order] = interpolated
        flags[:] = 1
        flags[valid] = 0

    return filled, flags


def geometry_quality_flags(values, default_used):
    numeric = np.asarray(values, dtype=float)
    valid = np.isfinite(numeric) & (numeric > 0)
    if default_used:
        flags = np.where(valid, 1, 2)
    else:
        flags = np.where(valid, 0, 2)
    return flags.astype(np.int8)


def prepare_auxiliary_data(normalized, ranges, progress_callback=None):
    validate_ranges(ranges)
    reject_reused_measurement_ids(normalized)

    pieces = []
    grouped = normalized.groupby(
        ["chamber_id", "measurement_id"], sort=False, dropna=False
    )
    total = grouped.ngroups
    for index, ((chamber_id, measurement_id), group) in enumerate(grouped, start=1):
        emit_progress(
            progress_callback, index - 1, total,
            f"Preparing {chamber_id}/{measurement_id}",
        )
        prepared_group = group.copy().sort_values("timestamp")
        elapsed = prepared_group["elapsed_seconds"].to_numpy(dtype=float)

        for output_name, spec in ENVIRONMENT_SPECS.items():
            filled, flags = interpolate_environment_signal(
                elapsed,
                prepared_group[spec["source"]].to_numpy(dtype=float),
                *ranges[output_name],
            )
            prepared_group[output_name] = filled
            prepared_group[spec["flag"]] = flags

        area_default = bool(prepared_group["area_default_used"].iloc[0])
        volume_default = bool(prepared_group["volume_default_used"].iloc[0])
        prepared_group["chamber_area_quality_flag"] = geometry_quality_flags(
            prepared_group["chamber_area_cm2"], area_default
        )
        prepared_group["chamber_volume_quality_flag"] = geometry_quality_flags(
            prepared_group["chamber_volume_cm3"], volume_default
        )

        valid_water = (
            prepared_group["pressure_quality_flag"].ne(2)
            & prepared_group["temperature_quality_flag"].ne(2)
            & prepared_group["relative_humidity_quality_flag"].ne(2)
        )
        water = np.full(len(prepared_group), np.nan, dtype=float)
        if valid_water.any():
            indexes = valid_water.to_numpy()
            water[indexes] = mole_fraction_water_vapor(
                prepared_group.loc[valid_water, "temperature_c"].to_numpy(dtype=float),
                prepared_group.loc[valid_water, "relative_humidity_percent"].to_numpy(dtype=float),
                prepared_group.loc[valid_water, "pressure_kpa"].to_numpy(dtype=float),
            )
        prepared_group["water_vapor_mmol_mol"] = water

        original_inputs = (
            prepared_group["pressure_quality_flag"].eq(0)
            & prepared_group["temperature_quality_flag"].eq(0)
            & prepared_group["relative_humidity_quality_flag"].eq(0)
        )
        prepared_group["water_vapor_quality_flag"] = np.where(
            ~valid_water, 2, np.where(original_inputs, 0, 1)
        ).astype(np.int8)

        wholly_invalid = []
        for output_name, spec in ENVIRONMENT_SPECS.items():
            if prepared_group[spec["flag"]].eq(2).all():
                wholly_invalid.append(spec["label"])
        invalid_geometry = []
        if prepared_group["chamber_area_quality_flag"].eq(2).any():
            invalid_geometry.append("chamber area")
        if prepared_group["chamber_volume_quality_flag"].eq(2).any():
            invalid_geometry.append("chamber volume")

        flag_columns = [
            "pressure_quality_flag",
            "temperature_quality_flag",
            "relative_humidity_quality_flag",
            "water_vapor_quality_flag",
            "chamber_area_quality_flag",
            "chamber_volume_quality_flag",
        ]
        all_flags = prepared_group[flag_columns].to_numpy(dtype=np.int8)
        if wholly_invalid or invalid_geometry or np.any(all_flags == 2):
            quality_flag = 2
            status = "invalid_auxiliary"
            details = []
            if wholly_invalid:
                details.append("Wholly invalid required variables: " + ", ".join(wholly_invalid))
            if invalid_geometry:
                details.append("Invalid mapped geometry: " + ", ".join(invalid_geometry))
            detail = "; ".join(details) or (
                "At least one required auxiliary value is unresolved or invalid."
            )
        elif np.any(all_flags == 1):
            quality_flag = 1
            status = "usable_with_fills_or_defaults"
            detail = "One or more values were interpolated, edge-filled, or supplied by geometry defaults."
        else:
            quality_flag = 0
            status = "valid_original"
            detail = "All required values are original and valid."

        prepared_group["measurement_quality_flag"] = np.int8(quality_flag)
        prepared_group["measurement_status"] = status
        prepared_group["measurement_status_detail"] = detail
        pieces.append(prepared_group)
        emit_progress(
            progress_callback, index, total,
            f"Prepared {index}/{total} measurements",
        )

    prepared = pd.concat(pieces, ignore_index=True)
    prepared = prepared.sort_values(
        ["chamber_id", "measurement_start", "measurement_id", "timestamp"]
    ).reset_index(drop=True)
    prepared.attrs.update(normalized.attrs)
    prepared.attrs["valid_ranges"] = ranges
    return prepared


def summarize_auxiliary_quality(prepared):
    summary = (
        prepared.groupby(["chamber_id", "measurement_id"], sort=True)
        .agg(
            start=("measurement_start", "first"),
            end=("timestamp", "max"),
            observations=("timestamp", "size"),
            status=("measurement_status", "first"),
            status_detail=("measurement_status_detail", "first"),
            pressure_filled=("pressure_quality_flag", lambda s: int((s == 1).sum())),
            pressure_invalid=("pressure_quality_flag", lambda s: int((s == 2).sum())),
            temperature_filled=("temperature_quality_flag", lambda s: int((s == 1).sum())),
            temperature_invalid=("temperature_quality_flag", lambda s: int((s == 2).sum())),
            rh_filled=("relative_humidity_quality_flag", lambda s: int((s == 1).sum())),
            rh_invalid=("relative_humidity_quality_flag", lambda s: int((s == 2).sum())),
        )
        .reset_index()
    )
    summary["start_date"] = summary["start"].dt.strftime("%Y-%m-%d")
    return summary


def plot_auxiliary_measurement(prepared, chamber_id, measurement_id):
    selected = prepared[
        prepared["chamber_id"].astype(str).eq(str(chamber_id))
        & prepared["measurement_id"].astype(str).eq(str(measurement_id))
    ].sort_values("elapsed_seconds")
    if selected.empty:
        raise ValueError("The selected chamber/measurement is not present.")

    fig, axes = plt.subplots(3, 1, figsize=(9, 8), dpi=110, sharex=True)
    plot_specs = [
        ("pressure_source", "pressure_kpa", "Pressure [kPa]"),
        ("temperature_source", "temperature_c", "Temperature [degree C]"),
        ("relative_humidity_source", "relative_humidity_percent", "Relative humidity [%]"),
    ]
    x = selected["elapsed_seconds"]
    for ax, (before, after, label) in zip(axes, plot_specs):
        ax.plot(x, selected[before], "o", color="#9a9a9a", alpha=0.7, label="Before QC")
        ax.plot(x, selected[after], "-o", color="#1769aa", markersize=3, label="After QC")
        ax.set_ylabel(label)
        ax.grid(alpha=0.25)
        ax.legend(loc="best")
    axes[-1].set_xlabel("Elapsed time [s]")
    status = selected["measurement_status"].iloc[0]
    fig.suptitle(f"{chamber_id} / {measurement_id}: {status}")
    fig.tight_layout()
    return fig


## NetCDF export

Each complete measurement is assigned to its start date, so a closure crossing midnight is not split. Output files use a flat observation dimension and repeat the measurement-level status columns for every observation.


In [4]:
def safe_identifier(value):
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip())
    text = text.strip("._-")
    return text or "chamber"


def ensure_unique_safe_chamber_ids(values):
    result = {}
    used = {}
    for value in sorted({str(value) for value in values}):
        safe = safe_identifier(value)
        if safe in used and used[safe] != value:
            raise ValueError(
                f"Chamber IDs {used[safe]!r} and {value!r} both map to filename stem {safe!r}."
            )
        used[safe] = value
        result[value] = safe
    return result


def build_auxiliary_dataset(group, chamber_id, file_date):
    group = group.sort_values(
        ["measurement_start", "measurement_id", "timestamp"]
    ).reset_index(drop=True)
    observation = np.arange(len(group), dtype=np.int64)

    data_vars = {
        "pressure_kpa": ("observation", group["pressure_kpa"].to_numpy(dtype=float)),
        "temperature_c": ("observation", group["temperature_c"].to_numpy(dtype=float)),
        "relative_humidity_percent": (
            "observation", group["relative_humidity_percent"].to_numpy(dtype=float)
        ),
        "water_vapor_mmol_mol": (
            "observation", group["water_vapor_mmol_mol"].to_numpy(dtype=float)
        ),
        "chamber_area_cm2": (
            "observation", group["chamber_area_cm2"].to_numpy(dtype=float)
        ),
        "chamber_volume_cm3": (
            "observation", group["chamber_volume_cm3"].to_numpy(dtype=float)
        ),
        "pressure_quality_flag": (
            "observation", group["pressure_quality_flag"].to_numpy(dtype=np.int8)
        ),
        "temperature_quality_flag": (
            "observation", group["temperature_quality_flag"].to_numpy(dtype=np.int8)
        ),
        "relative_humidity_quality_flag": (
            "observation", group["relative_humidity_quality_flag"].to_numpy(dtype=np.int8)
        ),
        "water_vapor_quality_flag": (
            "observation", group["water_vapor_quality_flag"].to_numpy(dtype=np.int8)
        ),
        "chamber_area_quality_flag": (
            "observation", group["chamber_area_quality_flag"].to_numpy(dtype=np.int8)
        ),
        "chamber_volume_quality_flag": (
            "observation", group["chamber_volume_quality_flag"].to_numpy(dtype=np.int8)
        ),
        "measurement_quality_flag": (
            "observation", group["measurement_quality_flag"].to_numpy(dtype=np.int8)
        ),
        "measurement_status": (
            "observation", group["measurement_status"].astype(str).to_numpy()
        ),
        "measurement_status_detail": (
            "observation", group["measurement_status_detail"].astype(str).to_numpy()
        ),
    }

    coords = {
        "observation": observation,
        "timestamp": (
            "observation",
            group["timestamp"].dt.tz_convert(None).to_numpy(dtype="datetime64[ns]"),
        ),
        "elapsed_seconds": (
            "observation", group["elapsed_seconds"].to_numpy(dtype=float)
        ),
        "measurement_id": (
            "observation", group["measurement_id"].astype(str).to_numpy()
        ),
        "chamber_id": str(chamber_id),
        "file_date": str(file_date),
    }
    dataset = xr.Dataset(data_vars=data_vars, coords=coords)

    variable_attrs = {
        "pressure_kpa": {"long_name": "chamber air pressure", "units": "kPa"},
        "temperature_c": {"long_name": "chamber air temperature", "units": "degree_Celsius"},
        "relative_humidity_percent": {"long_name": "relative humidity", "units": "percent"},
        "water_vapor_mmol_mol": {
            "long_name": "water vapor mole fraction derived with the Buck equation",
            "units": "mmol mol-1",
        },
        "chamber_area_cm2": {"long_name": "chamber footprint area", "units": "cm2"},
        "chamber_volume_cm3": {"long_name": "chamber volume", "units": "cm3"},
        "elapsed_seconds": {
            "long_name": "seconds since the start of this measurement",
            "units": "s",
        },
        "timestamp": {"long_name": "absolute observation timestamp in UTC"},
        "measurement_id": {"long_name": "closure measurement identifier"},
        "measurement_quality_flag": {
            "long_name": "measurement-level auxiliary data quality",
            "flag_values": np.array([0, 1, 2], dtype=np.int8),
            "flag_meanings": "valid_original usable_with_fills_or_defaults invalid_auxiliary",
        },
    }
    for variable, attrs in variable_attrs.items():
        dataset[variable].attrs.update(attrs)

    for variable in [
        "pressure_quality_flag",
        "temperature_quality_flag",
        "relative_humidity_quality_flag",
    ]:
        dataset[variable].attrs.update({
            "flag_values": np.array([0, 1, 2], dtype=np.int8),
            "flag_meanings": "original_valid interpolated_or_edge_filled invalid_unresolved",
        })
    dataset["water_vapor_quality_flag"].attrs.update({
        "flag_values": np.array([0, 1, 2], dtype=np.int8),
        "flag_meanings": "derived_from_original derived_from_filled invalid",
    })
    for variable in ["chamber_area_quality_flag", "chamber_volume_quality_flag"]:
        dataset[variable].attrs.update({
            "flag_values": np.array([0, 1, 2], dtype=np.int8),
            "flag_meanings": "mapped_valid default_supplied invalid",
        })

    ranges = group.attrs.get("valid_ranges", {})
    dataset.attrs.update({
        "title": "Auxiliary chamber data for later soil-gas flux calculation",
        "auxiliary_schema_version": SCHEMA_VERSION,
        "chamber_id": str(chamber_id),
        "file_date": str(file_date),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_files": json.dumps(group.attrs.get("source_files", [])),
        "interpolation_policy": (
            "Independent within chamber and measurement using elapsed time: "
            "linear interpolation for internal gaps, nearest valid anchor at edges; "
            "signals with no valid anchor remain missing."
        ),
        "valid_ranges": json.dumps({
            key: {"minimum": float(value[0]), "maximum": float(value[1])}
            for key, value in ranges.items()
        }),
        "environment_flag_definition": FLAG_MEANINGS["environment"],
        "water_vapor_flag_definition": FLAG_MEANINGS["water_vapor"],
        "geometry_flag_definition": FLAG_MEANINGS["geometry"],
        "measurement_flag_definition": FLAG_MEANINGS["measurement"],
        "geometry_default_area_cm2": DEFAULT_AREA_CM2,
        "geometry_default_volume_cm3": DEFAULT_VOLUME_CM3,
        "contains_dcdt": "false",
        "contains_flux": "false",
    })
    return dataset


def export_auxiliary_netcdf(
    prepared, output_folder, overwrite=False, progress_callback=None
):
    if prepared is None or prepared.empty:
        raise ValueError("Prepare the auxiliary data before export.")

    output_folder = sanitize_input_path(output_folder)
    chamber_names = ensure_unique_safe_chamber_ids(prepared["chamber_id"])
    groups = list(prepared.groupby(
        ["chamber_id", "measurement_start_date"], sort=True, dropna=False
    ))
    targets = [
        output_folder / f"{chamber_names[str(chamber_id)]}_{file_date}_auxiliary.nc"
        for (chamber_id, file_date), _ in groups
    ]
    existing = [path for path in targets if path.exists()]
    if existing and not overwrite:
        names = ", ".join(path.name for path in existing[:5])
        raise FileExistsError(
            f"{len(existing)} output file(s) already exist. Enable overwrite to replace them. "
            f"Examples: {names}"
        )

    output_folder.mkdir(parents=True, exist_ok=True)
    written = []
    total = len(groups)
    for index, (((chamber_id, file_date), group), target) in enumerate(
        zip(groups, targets), start=1
    ):
        emit_progress(
            progress_callback, index - 1, total, f"Writing {target.name}"
        )
        group = group.copy()
        group.attrs.update(prepared.attrs)
        dataset = build_auxiliary_dataset(group, chamber_id, file_date)
        try:
            dataset.to_netcdf(target)
        finally:
            dataset.close()
        written.append(target)
        emit_progress(
            progress_callback, index, total, f"Wrote {index}/{total} NetCDF files"
        )
    return written


def export_counts(prepared):
    measurements = prepared.drop_duplicates(["chamber_id", "measurement_id"])
    counts = measurements["measurement_status"].value_counts()
    return {
        "valid": int(counts.get("valid_original", 0)),
        "filled": int(counts.get("usable_with_fills_or_defaults", 0)),
        "invalid": int(counts.get("invalid_auxiliary", 0)),
    }


## Interactive workflow

Load the input first, review the automatically guessed mappings, prepare the QC data, inspect measurements, and export.


In [ ]:
state = {
    "raw": None,
    "normalized": None,
    "prepared": None,
    "updating": False,
}

style = {"description_width": "165px"}
medium = widgets.Layout(width="410px")
wide = widgets.Layout(width="820px")
path_layout = widgets.Layout(width="820px", height="70px")
task_progress_widget = widgets.IntProgress(
    value=0, min=0, max=100, description="Idle", bar_style="",
    style={"description_width": "190px"}, layout=wide,
)

input_mode_widget = widgets.Dropdown(
    options=["CSV", "JSON folder"], value="CSV", description="Input mode",
    style=style, layout=medium
)
folder_filter_widget = widgets.Checkbox(
    value=False, description="Filter folder", indent=False,
    layout=widgets.Layout(width="145px")
)
folder_name_widget = widgets.Text(
    value="", description="Folder name", placeholder="e.g. 1-1",
    continuous_update=False, disabled=True,
    style={"description_width": "95px"}, layout=widgets.Layout(width="265px")
)
input_path_widget = widgets.Textarea(
    value="", description="Input path", placeholder="Paste a CSV file/folder or JSON folder",
    continuous_update=False, style=style, layout=path_layout
)
pattern_widget = widgets.Text(
    value="*.csv", description="CSV pattern", continuous_update=False,
    style=style, layout=medium
)
delimiter_widget = widgets.Text(
    value=",", description="CSV delimiter", continuous_update=False,
    style=style, layout=medium
)
load_button = widgets.Button(description="Load input", button_style="primary")
load_status = widgets.HTML(value="No input loaded.")

empty_options = [("Load input first", "")]
def column_dropdown(description):
    return widgets.Dropdown(
        options=empty_options, value="", description=description,
        disabled=True, style=style, layout=medium
    )

timestamp_widget = column_dropdown("Timestamp")
measurement_widget = column_dropdown("Measurement ID")
pressure_widget = column_dropdown("Pressure")
temperature_widget = column_dropdown("Temperature")
humidity_widget = column_dropdown("Relative humidity")
chamber_widget = column_dropdown("Chamber ID")
area_widget = column_dropdown("Area [cm2]")
volume_widget = column_dropdown("Volume [cm3]")

pressure_unit_widget = widgets.Dropdown(
    options=["Pa", "kPa"], value="Pa", description="Pressure unit",
    style=style, layout=medium
)
constant_chamber_widget = widgets.Text(
    value="", description="Constant chamber ID", continuous_update=False,
    style=style, layout=medium
)
default_area_widget = widgets.FloatText(
    value=DEFAULT_AREA_CM2, description="Default area [cm2]",
    style=style, layout=medium
)
default_volume_widget = widgets.FloatText(
    value=DEFAULT_VOLUME_CM3, description="Default volume [cm3]",
    style=style, layout=medium
)

pressure_min_widget = widgets.FloatText(
    value=80.0, description="Pressure min [kPa]", style=style, layout=medium
)
pressure_max_widget = widgets.FloatText(
    value=120.0, description="Pressure max [kPa]", style=style, layout=medium
)
temperature_min_widget = widgets.FloatText(
    value=-40.0, description="Temperature min [C]", style=style, layout=medium
)
temperature_max_widget = widgets.FloatText(
    value=60.0, description="Temperature max [C]", style=style, layout=medium
)
humidity_min_widget = widgets.FloatText(
    value=0.0, description="RH min [%]", style=style, layout=medium
)
humidity_max_widget = widgets.FloatText(
    value=100.0, description="RH max [%]", style=style, layout=medium
)
prepare_button = widgets.Button(
    description="Prepare + summarize", button_style="info", disabled=True
)
prepare_status = widgets.HTML(value="Load and map input before preparing.")
quality_output = widgets.Output()

preview_chamber_widget = widgets.Dropdown(
    options=[], description="Preview chamber", disabled=True,
    style=style, layout=medium
)
preview_measurement_widget = widgets.Dropdown(
    options=[], description="Preview measurement", disabled=True,
    style=style, layout=medium
)
preview_output = widgets.Output()

output_folder_widget = widgets.Textarea(
    value=str(DEFAULT_OUTPUT_DIR), description="Output folder",
    continuous_update=False, style=style, layout=path_layout
)
overwrite_widget = widgets.Checkbox(
    value=False, description="Overwrite existing files", indent=False, layout=medium
)
export_button = widgets.Button(
    description="Export NetCDF files", button_style="success", disabled=True
)
export_output = widgets.Output()

mapping_widgets = [
    timestamp_widget, measurement_widget, pressure_widget, temperature_widget,
    humidity_widget, chamber_widget, area_widget, volume_widget
]


def set_task_progress(value, description, bar_style="info"):
    task_progress_widget.value = min(max(int(round(value)), 0), 100)
    task_progress_widget.description = str(description)
    task_progress_widget.bar_style = bar_style


def reset_task_progress(change=None):
    set_task_progress(0, "Idle", "")


def begin_task(description):
    set_task_progress(5, description, "info")


def complete_task(description):
    set_task_progress(100, description, "success")


def fail_task(description):
    set_task_progress(100, description, "danger")


def measured_progress_callback(start, end):
    def update(completed, total, message):
        fraction = min(max(float(completed) / max(float(total), 1.0), 0.0), 1.0)
        set_task_progress(start + (end - start) * fraction, message, "info")
    return update


def show_exception(exc):
    print(f"{type(exc).__name__}: {exc}")


def clear_prepared():
    state["normalized"] = None
    state["prepared"] = None
    prepare_status.value = "Review mappings and valid ranges, then prepare."
    export_button.disabled = True
    preview_chamber_widget.options = []
    preview_chamber_widget.disabled = True
    preview_measurement_widget.options = []
    preview_measurement_widget.disabled = True
    with quality_output:
        clear_output(wait=True)
    with preview_output:
        clear_output(wait=True)


def clear_loaded_input(status="No input loaded."):
    reset_task_progress()
    state["raw"] = None
    clear_prepared()
    state["updating"] = True
    try:
        for widget in mapping_widgets:
            widget.options = empty_options
            widget.value = ""
            widget.disabled = True
    finally:
        state["updating"] = False
    prepare_button.disabled = True
    prepare_status.value = "Load and map input before preparing."
    load_status.value = status


def invalidate_loaded_input(change=None):
    if state["updating"]:
        return
    clear_loaded_input("Input settings changed. Load input again.")


def on_folder_filter_toggled(change):
    folder_name_widget.disabled = not bool(change["new"])
    invalidate_loaded_input(change)


def set_column_options(columns):
    columns = list(columns)
    options = [""] + columns
    state["updating"] = True
    try:
        for widget in mapping_widgets:
            widget.options = options
            widget.disabled = False
        timestamp_widget.value = guess_column(columns, [
            "timestamp", "datetime", "datetime_utc", "date_time", "time"
        ])
        measurement_widget.value = guess_column(columns, [
            "measurement_id", "measurement", "id", "closure_id"
        ])
        pressure_widget.value = guess_column(columns, [
            "bmp_pressure", "pressure", "pressure_pa", "pressure_kpa"
        ])
        temperature_widget.value = guess_column(columns, [
            "si_temperature", "temperature", "temperature_c", "temp"
        ])
        humidity_widget.value = guess_column(columns, [
            "si_humidity", "relative_humidity", "humidity", "rh"
        ])
        chamber_widget.value = guess_column(columns, [
            "chamber_id", "chamber", "plot_id"
        ])
        area_widget.value = guess_column(columns, [
            "chamber_area_cm2", "area_cm2", "area"
        ])
        volume_widget.value = guess_column(columns, [
            "chamber_volume_cm3", "volume_cm3", "volume"
        ])
    finally:
        state["updating"] = False


def on_load_clicked(_):
    clear_loaded_input("Loading input...")
    load_button.disabled = True
    begin_task("Loading input")
    try:
        folder_name = folder_name_widget.value if folder_filter_widget.value else None
        set_task_progress(10, "Discovering input files", "info")
        raw = load_auxiliary_input(
            input_mode_widget.value,
            input_path_widget.value,
            pattern=pattern_widget.value,
            delimiter=delimiter_widget.value,
            folder_name=folder_name,
            progress_callback=measured_progress_callback(20, 90),
        )
        set_task_progress(95, "Finalizing loaded input", "info")
        state["raw"] = raw
        set_column_options(raw.columns)
        prepare_button.disabled = False
        ignored = len(raw.attrs.get("ignored_files", []))
        matched_count = len(raw.attrs.get("matched_folders", []))
        filter_status = (
            ""
            if folder_name is None
            else (
                f" Folder filter {folder_name.strip()!r} matched "
                f"{matched_count} folder(s)."
            )
        )
        load_status.value = (
            f"<b>Loaded:</b> {len(raw):,} rows, {len(raw.columns):,} columns, "
            f"{len(raw.attrs.get('source_files', [])):,} source file(s). "
            f"Ignored small JSON files: {ignored}.{filter_status}"
        )
        complete_task("Load complete")
    except Exception as exc:
        clear_loaded_input(f"<b>Load failed:</b> {type(exc).__name__}: {exc}")
        fail_task("Load failed")
    finally:
        load_button.disabled = False


def current_ranges():
    return {
        "pressure_kpa": (
            float(pressure_min_widget.value), float(pressure_max_widget.value)
        ),
        "temperature_c": (
            float(temperature_min_widget.value), float(temperature_max_widget.value)
        ),
        "relative_humidity_percent": (
            float(humidity_min_widget.value), float(humidity_max_widget.value)
        ),
    }


def refresh_preview_measurements(change=None):
    prepared = state["prepared"]
    if prepared is None or preview_chamber_widget.value is None:
        return
    chamber = str(preview_chamber_widget.value)
    values = (
        prepared.loc[prepared["chamber_id"].astype(str).eq(chamber), "measurement_id"]
        .astype(str).drop_duplicates().tolist()
    )
    state["updating"] = True
    try:
        preview_measurement_widget.options = values
        preview_measurement_widget.disabled = not bool(values)
    finally:
        state["updating"] = False


def display_preview(change=None):
    if state["updating"] or state["prepared"] is None:
        return
    chamber = preview_chamber_widget.value
    measurement = preview_measurement_widget.value
    if chamber is None or measurement is None:
        return
    with preview_output:
        clear_output(wait=True)
        try:
            fig = plot_auxiliary_measurement(state["prepared"], chamber, measurement)
            display(fig)
            plt.close(fig)
            selected = state["prepared"][
                state["prepared"]["chamber_id"].astype(str).eq(str(chamber))
                & state["prepared"]["measurement_id"].astype(str).eq(str(measurement))
            ]
            print(selected["measurement_status_detail"].iloc[0])
        except Exception as exc:
            show_exception(exc)


def on_prepare_clicked(_):
    prepare_button.disabled = True
    begin_task("Preparing data")
    with quality_output:
        clear_output(wait=True)
        try:
            if state["raw"] is None:
                raise ValueError("Load input before preparing.")
            set_task_progress(10, "Normalizing input", "info")
            normalized = normalize_auxiliary_input(
                state["raw"],
                timestamp_col=timestamp_widget.value,
                measurement_id_col=measurement_widget.value,
                pressure_col=pressure_widget.value,
                pressure_unit=pressure_unit_widget.value,
                temperature_col=temperature_widget.value,
                humidity_col=humidity_widget.value,
                chamber_id_col=chamber_widget.value,
                constant_chamber_id=constant_chamber_widget.value,
                area_col=area_widget.value,
                volume_col=volume_widget.value,
                default_area_cm2=float(default_area_widget.value),
                default_volume_cm3=float(default_volume_widget.value),
            )
            ranges = current_ranges()
            set_task_progress(20, "Preparing measurements", "info")
            prepared = prepare_auxiliary_data(
                normalized, ranges,
                progress_callback=measured_progress_callback(20, 80),
            )
            state["normalized"] = normalized
            state["prepared"] = prepared

            set_task_progress(85, "Summarizing quality", "info")
            summary = summarize_auxiliary_quality(prepared)
            display(summary)
            counts = export_counts(prepared)
            print(
                f"Measurements: {len(summary)} total; "
                f"{counts['valid']} valid, {counts['filled']} usable with fills/defaults, "
                f"{counts['invalid']} invalid but retained."
            )

            chambers = prepared["chamber_id"].astype(str).drop_duplicates().tolist()
            state["updating"] = True
            try:
                preview_chamber_widget.options = chambers
                preview_chamber_widget.disabled = not bool(chambers)
            finally:
                state["updating"] = False
            refresh_preview_measurements()
            export_button.disabled = False
            prepare_status.value = (
                f"<b>Prepared:</b> {len(prepared):,} observations across "
                f"{len(summary):,} measurements."
            )
            set_task_progress(95, "Building preview", "info")
            display_preview()
            complete_task("Preparation complete")
        except Exception as exc:
            state["normalized"] = None
            state["prepared"] = None
            export_button.disabled = True
            prepare_status.value = f"<b>Preparation failed:</b> {type(exc).__name__}: {exc}"
            fail_task("Preparation failed")
            show_exception(exc)
        finally:
            prepare_button.disabled = state["raw"] is None


def on_export_clicked(_):
    export_button.disabled = True
    begin_task("Exporting NetCDF")
    with export_output:
        clear_output(wait=True)
        try:
            set_task_progress(10, "Validating export", "info")
            written = export_auxiliary_netcdf(
                state["prepared"],
                output_folder_widget.value,
                overwrite=overwrite_widget.value,
                progress_callback=measured_progress_callback(15, 95),
            )
            counts = export_counts(state["prepared"])
            print(f"Wrote {len(written)} NetCDF file(s):")
            for path in written:
                print(f"  {path}")
            print(
                f"Measurement counts: {counts['valid']} valid, "
                f"{counts['filled']} usable with fills/defaults, "
                f"{counts['invalid']} invalid_auxiliary."
            )
            complete_task("Export complete")
        except Exception as exc:
            fail_task("Export failed")
            show_exception(exc)
        finally:
            export_button.disabled = state["prepared"] is None


def invalidate_prepared(change=None):
    if state["updating"] or state["raw"] is None:
        return
    clear_prepared()
    reset_task_progress()


load_button.on_click(on_load_clicked)
prepare_button.on_click(on_prepare_clicked)
export_button.on_click(on_export_clicked)
preview_chamber_widget.observe(refresh_preview_measurements, names="value")
preview_chamber_widget.observe(display_preview, names="value")
preview_measurement_widget.observe(display_preview, names="value")
for widget in mapping_widgets + [
    pressure_unit_widget,
    constant_chamber_widget,
    default_area_widget,
    default_volume_widget,
    pressure_min_widget,
    pressure_max_widget,
    temperature_min_widget,
    temperature_max_widget,
    humidity_min_widget,
    humidity_max_widget,
]:
    widget.observe(invalidate_prepared, names="value")
for widget in [
    input_mode_widget,
    input_path_widget,
    folder_name_widget,
    pattern_widget,
    delimiter_widget,
]:
    widget.observe(invalidate_loaded_input, names="value")
folder_filter_widget.observe(on_folder_filter_toggled, names="value")
output_folder_widget.observe(reset_task_progress, names="value")
overwrite_widget.observe(reset_task_progress, names="value")

progress_box = widgets.VBox([
    widgets.HTML("<h3>Task progress</h3>"),
    task_progress_widget,
])

load_box = widgets.VBox([
    widgets.HTML("<h3>1. Load source data</h3>"),
    widgets.HBox([input_mode_widget, folder_filter_widget, folder_name_widget]),
    input_path_widget,
    widgets.HBox([pattern_widget, delimiter_widget]),
    load_button,
    load_status,
])

mapping_box = widgets.VBox([
    widgets.HTML("<h3>2. Map auxiliary columns</h3>"),
    widgets.HTML(
        "Timestamp and measurement ID are required. Map chamber ID or enter a constant. "
        "Leave geometry mappings blank to use the defaults."
    ),
    widgets.HBox([timestamp_widget, measurement_widget]),
    widgets.HBox([pressure_widget, pressure_unit_widget]),
    widgets.HBox([temperature_widget, humidity_widget]),
    widgets.HBox([chamber_widget, constant_chamber_widget]),
    widgets.HBox([area_widget, default_area_widget]),
    widgets.HBox([volume_widget, default_volume_widget]),
])

quality_box = widgets.VBox([
    widgets.HTML("<h3>3. Validate, interpolate, and preview quality</h3>"),
    widgets.HBox([pressure_min_widget, pressure_max_widget]),
    widgets.HBox([temperature_min_widget, temperature_max_widget]),
    widgets.HBox([humidity_min_widget, humidity_max_widget]),
    prepare_button,
    prepare_status,
    quality_output,
    widgets.HBox([preview_chamber_widget, preview_measurement_widget]),
    preview_output,
])

export_box = widgets.VBox([
    widgets.HTML("<h3>4. Export one file per chamber and start day</h3>"),
    output_folder_widget,
    overwrite_widget,
    export_button,
    export_output,
])

display(widgets.VBox([progress_box, load_box, mapping_box, quality_box, export_box]))
